In [1]:
import copy
import random
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, recall_score

In [14]:
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [15]:
data = load_breast_cancer()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print(X_train.shape, X_test.shape)

(455, 30) (114, 30)


In [16]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

train_input = torch.tensor(X_train, dtype=torch.float32, device=device)
train_label = torch.tensor(y_train, dtype=torch.long, device=device)
test_input = torch.tensor(X_test, dtype=torch.float32, device=device)
test_label = torch.tensor(y_test, dtype=torch.long, device=device)

In [17]:
input_size = 30
hidden_1 = 10
hidden_2 = 5
output_size = 2
degree = 4

In [18]:
c1 = nn.Parameter(torch.empty(input_size, hidden_1, degree + 1, device=device))
b1 = nn.Parameter(torch.zeros(hidden_1, device=device))

c2 = nn.Parameter(torch.empty(hidden_1, hidden_2, degree + 1, device=device))
b2 = nn.Parameter(torch.zeros(hidden_2, device=device))

c3 = nn.Parameter(torch.empty(hidden_2, output_size, degree + 1, device=device))
b3 = nn.Parameter(torch.zeros(output_size, device=device))

nn.init.xavier_normal_(c1)
nn.init.xavier_normal_(c2)
nn.init.xavier_normal_(c3)

Parameter containing:
tensor([[[-0.2835,  0.1544,  0.3345,  0.0128, -0.0707],
         [-0.0678,  0.0046, -0.0250, -0.0614, -0.1143]],

        [[-0.2499, -0.1637, -0.2181,  0.0720, -0.0470],
         [-0.4149,  0.1060, -0.1428, -0.0253,  0.2227]],

        [[ 0.0777,  0.0110,  0.0006,  0.2385,  0.3414],
         [ 0.2357,  0.3116, -0.1735,  0.1040, -0.3306]],

        [[-0.1274, -0.3134, -0.3110, -0.1830, -0.0228],
         [-0.0822, -0.3139, -0.1137,  0.2893,  0.4776]],

        [[-0.0952, -0.3215,  0.1521,  0.2414,  0.1347],
         [-0.0406,  0.4046,  0.3453, -0.2782, -0.4100]]], requires_grad=True)

In [19]:
parameters = [c1, b1, c2, b2, c3, b3]

In [20]:
def make_chebyshev_polynomials(x, degree):
    x = torch.tanh(x)

    polynomials = [torch.ones_like(x)]

    if degree >= 1:
        polynomials.append(x)

    for n in range(2, degree + 1):
        next_polynomial = 2 * x * polynomials[-1] - polynomials[-2]
        polynomials.append(next_polynomial)

    return torch.stack(polynomials, dim=2)

In [21]:
def chebyshev_layer(x, coefficients, bias):
    layer_degree = coefficients.shape[2] - 1
    chebyshev_values = make_chebyshev_polynomials(x, layer_degree)
    output = torch.einsum('bid,iod->bo', chebyshev_values, coefficients)
    return output + bias

In [22]:
def forward(x):
    x = chebyshev_layer(x, c1, b1)
    x = nn.functional.layer_norm(x, (hidden_1,))

    x = chebyshev_layer(x, c2, b2)
    x = nn.functional.layer_norm(x, (hidden_2,))

    x = chebyshev_layer(x, c3, b3)
    return x

In [23]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(parameters, lr=0.005, weight_decay=0.0004)
l1_strength = 1e-6

In [24]:
max_epochs = 3000
patience = 150
best_loss = float('inf')
best_parameters = None
patience_counter = 0

for epoch in range(max_epochs):
    train_logits = forward(train_input)
    classification_loss = criterion(train_logits, train_label)
    l1_penalty = c1.abs().sum() + c2.abs().sum() + c3.abs().sum()
    loss = classification_loss + l1_strength * l1_penalty

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        test_logits = forward(test_input)
        test_loss = criterion(test_logits, test_label).item()

    if test_loss < best_loss - 1e-6:
        best_loss = test_loss
        best_parameters = [p.detach().clone() for p in parameters]
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch % 100 == 0:
        print(
            f'Epoch {epoch:4d} | '
            f'train loss {classification_loss.item():.4f} | '
            f'test loss {test_loss:.4f}'
        )

    if patience_counter >= patience:
        print('Early stopping at epoch', epoch)
        break

with torch.no_grad():
    for parameter, best_value in zip(parameters, best_parameters):
        parameter.copy_(best_value)


Epoch    0 | train loss 0.7849 | test loss 0.6062
Epoch  100 | train loss 0.0039 | test loss 0.1821
Early stopping at epoch 194


In [25]:
with torch.no_grad():
    test_logits = forward(test_input)
    probabilities = torch.softmax(test_logits, dim=1)
    predictions = probabilities.argmax(dim=1)

y_true = test_label.cpu().numpy()
y_pred = predictions.cpu().numpy()

print(classification_report(y_true, y_pred, target_names=data.target_names))
print('Recall:', recall_score(y_true, y_pred, zero_division=0))


              precision    recall  f1-score   support

   malignant       0.97      0.93      0.95        42
      benign       0.96      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114

Recall: 0.9861111111111112


In [26]:
with torch.no_grad():
    feature_strength = c1.abs().sum(dim=(1, 2)).cpu().numpy()

ranking = np.argsort(feature_strength)[::-1]
for index in ranking[:10]:
    print(f'{data.feature_names[index]:30s} {feature_strength[index]:.4f}')


worst concave points           5.7051
symmetry error                 5.3024
worst compactness              5.1290
worst texture                  4.9224
worst smoothness               4.9077
radius error                   4.7913
area error                     4.7562
mean symmetry                  4.7361
worst concavity                4.6621
mean concave points            4.6587


In [27]:
def print_edge_equation(coefficients, input_index, output_index):
    values = (
        coefficients[input_index, output_index]
        .detach()
        .cpu()
        .numpy()
    )

    equation = (
        f"{values[0]:+.4f} T0(z) "
        f"{values[1]:+.4f} T1(z) "
        f"{values[2]:+.4f} T2(z) "
        f"{values[3]:+.4f} T3(z) "
        f"{values[4]:+.4f} T4(z)"
    )

    print(equation)
    print("where z = tanh(x)")

In [28]:
print_edge_equation(
    c1,
    input_index=0,
    output_index=0
)

-0.1597 T0(z) -0.2014 T1(z) -0.0325 T2(z) -0.0738 T3(z) +0.1798 T4(z)
where z = tanh(x)


In [29]:
def print_expanded_edge(coefficients, input_index, output_index):
    c = (
        coefficients[input_index, output_index]
        .detach()
        .cpu()
    )

    constant = c[0] - c[2] + c[4]
    linear = c[1] - 3 * c[3]
    quadratic = 2 * c[2] - 8 * c[4]
    cubic = 4 * c[3]
    quartic = 8 * c[4]

    print(
        f"φ(z) = "
        f"{constant.item():+.4f} "
        f"{linear.item():+.4f}z "
        f"{quadratic.item():+.4f}z² "
        f"{cubic.item():+.4f}z³ "
        f"{quartic.item():+.4f}z⁴"
    )

    print("where z = tanh(x)")

In [30]:
print_expanded_edge(
    c1,
    input_index=0,
    output_index=0
)

φ(z) = +0.0527 +0.0199z -1.5039z² -0.2950z³ +1.4388z⁴
where z = tanh(x)
